# HumanML3D 数据集产生的详细解释

# 学习顺序：
- （1）先看step1，以及对应的代码（就着github的ipynb）；
- （2）然后跑step1_processAMASS.py 这个脚本，可以debug学习一下；
- （3）接下来可视化某个处理完之后的文件，使用step_12_visual_motion.py这个脚本；
- （4）继续看step2，以及对应的代码（就着github的ipynb）；

## step 1： raw_pose_processing.ipynb ：https://github.com/EricGuo5513/HumanML3D/blob/main/raw_pose_processing.ipynb

这是一个非常好的切入点。HumanML3D 是文本生成动作（Text-to-Motion）领域非常经典的工作，它提出的这套动作特征提取流程（Motion Representation）几乎成为了后来很多工作（如 MotionDiffuse, MDM, T2M-GPT 等）的标准配置。

你提供的这个 `.ipynb` 脚本是数据处理的第一阶段：**从原始的 AMASS 数据集（SMPL 参数）中提取 3D 骨骼关键点（Joints）**。

请注意：**这个脚本输出的还不是 263 维的特征向量，而是 (N, 22, 3) 的原始 3D 坐标数据。** 263 维特征是在这之后计算出来的。

下面我将分为三部分为你详细解答：
1.  **脚本代码断案**：逐行分析这个 Notebook 在做什么，输入输出是什么。
2.  **揭秘 263 维**：结合 HumanML3D 的后续处理逻辑，详细拆解这 263 个维度分别是真的什么。
3.  **实战改造**：编写一个批处理脚本，针对单个 AMASS zip 文件进行转换，并加入可视化环节。

---

### 第一部分：代码断案 - `raw_pose_processing.ipynb` 在做什么？

这个脚本的核心目的是：**SMPL 参数 $\rightarrow$ 3D 关节坐标**。

#### 1. 环境与模型加载 (Cell 1-3)
*   **输入**：
    *   `male_bm_path` / `female_bm_path`：这是 SMPL-H 的身体模型文件（需要去官网下载）。
*   **关键操作**：
    *   使用了 `human_body_prior` 库中的 `BodyModel`。
    *   初始化了男性和女性两个身体模型（Body Model），这是因为 AMASS 数据集里的动作数据是区分性别的。
    *   **维度关注**：`num_betas=10` (体型参数), `num_dmpls=8` (软组织参数)。

#### 2. 文件遍历 (Cell 4-6)
*   **操作**：遍历 `./amass_data` 文件夹，找到所有的 `.npz` 文件。这些是 AMASS 的原始数据文件。

#### 3. 核心转换函数 `amass_to_pose` (Cell 7 - 最重要！)
这是脚本的灵魂。让我们看看它对一个数据文件做了什么：

*   **输入**：`src_path` (一个 `.npz` 文件路径)。
*   **Step 1: 读取与降采样**
    *   读取 `mocap_framerate`（原始帧率，通常是 60, 120 等）。
    *   目标帧率 `ex_fps = 20`。HumanML3D 统一使用 20fps。
    *   计算 `down_sample` 步长，对数据进行切片 `[::down_sample]`。
*   **Step 2: 准备 SMPL 参数**
    *   `root_orient`: 根节点朝向 (Nx3)。
    *   `pose_body`: 身体关节旋转 (Nx63, 对应 21 个关节 $\times$ 3 轴角)。
    *   `pose_hand`: 手部参数 (Nx..., SMPL-H 特有，但在 HumanML3D 的 22 关节简化版中，手部细节通常不作为主要关节被保留，或者只保留手腕)。
    *   `trans`: 全局位移 (Nx3)。
    *   `betas`: 体型参数 (Nx10)。
*   **Step 3: Forward Pass (正向运动学)**
    *   `body = bm(**body_parms)`: 把参数喂给模型。
    *   **关键输出**: `body.Jtr`。这是模型的 **Joint Translation (关节位置)**。
    *   **维度**: `(N, 52, 3)` 或 `(N, 22, 3)`，取决于 `BodyModel` 的配置。通常 SMPL-H 输出很多关节，但 HumanML3D 取的是前 22 个主要关节。
*   **Step 4: 坐标系修正**
    *   `trans_matrix`: 这是一个旋转矩阵，用来修正坐标轴（比如把 Z-up 变成 Y-up）。
    *   `np.dot(pose_seq_np, trans_matrix)`: 应用旋转。
*   **输出**：保存为 `.npy` 文件，内容是 `(N, 22, 3)` 的 numpy 数组。

#### 4. 镜像与分割 (Cell 10-13)
*   **操作**：
    *   `swap_left_right`: 数据增强。将动作沿 X 轴镜像（左手变右手，右脚变左脚）。
    *   根据 `index.csv`（HumanAct12 数据集的定义文件）裁剪动作的 `start_frame` 和 `end_frame`。
    *   分别保存原始动作和镜像动作（文件名加 'M'）。

---

### 第二部分：揭秘 HumanML3D 的 263 维特征（后面还会再处理）

正如我刚才提到的，上面的脚本只输出了 **(N, 22, 3)** 的关节坐标 (Joint Positions)。
要变成 **263 维**，需要经过特征提取（Feature Extraction）。这通常在 `motion_process.py` 或类似文件中完成。

这 263 维特征向量 `(Frame, 263)` 包含的信息极其丰富，旨在让模型更容易学习动作的物理特性。

**公式拆解：263 = 4 + 63 + 66 + 126 + 4**

1.  **Root (根节点) 信息 [4维]**:
    *   **1维**: 根节点的 **角速度 (Angular Velocity)**，具体是沿 Y 轴（垂直轴）的旋转速度。告诉模型人转得有多快。
    *   **2维**: 根节点的 **线性速度 (Linear Velocity)**，在 X-Z 平面（地面）上的速度。告诉模型人走得有多快。
    *   **1维**: 根节点的 **高度 (Height)**，即 Y 轴坐标。告诉模型人是站着、蹲着还是跳起来。
    *   *注：这里去掉了根节点的绝对 X, Z 坐标和绝对朝向，为了让动作具有“平移不变性”。*

2.  **Local Joint Positions (局部关节位置) [63维]**:
    *   总共有 22 个关节。去掉根节点（因为是相对根节点的），剩下 21 个关节。
    *   每个关节有 (x, y, z) 3 个坐标。
    *   $21 \times 3 = 63$。
    *   这些坐标是**相对于根节点**的坐标，描述身体姿态。

3.  **Joint Velocities (关节速度) [66维]**:
    *   所有 22 个关节（包括根节点）的线性速度。
    *   $22 \times 3 = 66$。
    *   描述动作的动量和快慢。

4.  **Joint Rotations (关节旋转 - 6D) [126维]**:
    *   这是最占维度的部分。
    *   除了位置，模型还需要知道关节的旋转角度。
    *   HumanML3D 使用 **6D 旋转表示法 (6D Rotation Representation)**，比四元数更适合神经网络回归（连续性更好）。
    *   计算方法：通常是对根节点以外的 21 个关节的旋转矩阵进行转换。
    *   $21 \text{ (joints)} \times 6 \text{ (dim)} = 126$。

5.  **Foot Contacts (脚部接触标签) [4维]**:
    *   这是为了解决“滑步” (Foot Sliding) 问题。
    *   通常对应：左脚跟、左脚尖、右脚跟、右脚尖（或者左脚、右脚）。
    *   通过计算脚部速度是否接近 0 来判定。
    *   是一个二值（或 Sigmoid 概率）信号：1 表示着地，0 表示抬起。

**Sum: 4 + 63 + 66 + 126 + 4 = 263。破案！**

---

### 第三部分：实战 - 编写批处理转换脚本

现在，我将把上面的 Jupyter Notebook 改写成一个清晰的 Python 脚本。为了方便学习，这个脚本将：
1.  只处理一个 Zip 文件或文件夹。
2.  **增加可视化**：处理完一个动作后，画出骨架图，让你看到这 22 个点长什么样。

#### 准备工作
你需要确保安装了必要的库：
```bash
pip install torch numpy matplotlib smplx
```
*注：原始代码用了 `human_body_prior`，这里为了通用性，我将代码适配为通用的逻辑，或者你需要确保 `human_body_prior` 在你的 PYTHONPATH 下。为了演示完整性，我假设你有 `human_body_prior`，因为它是处理 AMASS 的官方推荐。*

#### 脚本代码：`convert_amass_single.py`

```python
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import argparse
from tqdm import tqdm

# 尝试导入 BodyModel，如果你的环境没有配置好 human_body_prior，这一步会报错
try:
    from human_body_prior.body_model.body_model import BodyModel
    from human_body_prior.tools.omni_tools import copy2cpu as c2c
except ImportError:
    print("Error: 请确保安装了 human_body_prior 库 (https://github.com/nghorbani/human_body_prior)")
    exit()

# ---------------- 配置区域 ----------------
# 请修改这里的路径为你本地的路径
MALE_BM_PATH = './body_models/smplh/male/model.npz'
FEMALE_BM_PATH = './body_models/smplh/female/model.npz'
TARGET_FPS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 22个关节的连接关系 (用于可视化连线)
# SMPL 22 joints indices usually follow kinematic tree
# 这是一个简化的连接示意，用于画火柴人
KINEMATIC_CHAIN = [
    [0, 1], [0, 2], [0, 3], [1, 4], [2, 5], [3, 6], [4, 7], [5, 8], [6, 9],
    [7, 10], [8, 11], [9, 12], [9, 13], [9, 14], [12, 15], [13, 16], [14, 17],
    [16, 18], [17, 19], [18, 20], [19, 21]
]

def load_body_models():
    """加载 SMPL-H 模型"""
    print(f"Loading Body Models on {DEVICE}...")
    male_bm = BodyModel(bm_fname=MALE_BM_PATH, num_betas=10, num_dmpls=8).to(DEVICE)
    female_bm = BodyModel(bm_fname=FEMALE_BM_PATH, num_betas=10, num_dmpls=8).to(DEVICE)
    return male_bm, female_bm

def visualize_pose(pose_data, save_path, frame_idx=0):
    """
    可视化某一帧的骨架并保存为图片
    Input: pose_data (T, 22, 3)
    """
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # 获取特定帧的数据
    frame_pose = pose_data[frame_idx] # (22, 3)
    
    # 绘制关节点
    ax.scatter(frame_pose[:, 0], frame_pose[:, 1], frame_pose[:, 2], c='r', marker='o')
    
    # 绘制骨骼连线
    for chain in KINEMATIC_CHAIN:
        # 注意：这里的连接关系可能需要根据 SMPL 的索引微调
        # 这里做一个简单的容错，防止索引越界（如果只有22个点）
        if chain[0] < frame_pose.shape[0] and chain[1] < frame_pose.shape[0]:
            x_lines = [frame_pose[chain[0], 0], frame_pose[chain[1], 0]]
            y_lines = [frame_pose[chain[0], 1], frame_pose[chain[1], 1]]
            z_lines = [frame_pose[chain[0], 2], frame_pose[chain[1], 2]]
            ax.plot(x_lines, y_lines, z_lines, c='b')

    # 设置轴标签
    ax.set_xlabel('X')
    ax.set_ylabel('Y (Up)')
    ax.set_zlabel('Z')
    
    # 为了保持比例一致
    limit = 1.0
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit) # Y轴可能需要根据数据调整
    ax.set_zlim(-limit, limit)
    
    plt.title(f"Visualized Frame {frame_idx} (22 Joints)")
    plt.savefig(save_path)
    plt.close()
    print(f"Visualization saved to {save_path}")

def process_single_file(npz_path, output_dir, male_bm, female_bm):
    """处理单个 .npz 文件"""
    filename = os.path.basename(npz_path).replace('.npz', '')
    
    # 1. 加载数据
    try:
        bdata = np.load(npz_path, allow_pickle=True)
    except Exception as e:
        print(f"Failed to load {npz_path}: {e}")
        return

    # 2. 检查基本信息
    try:
        fps = bdata['mocap_framerate']
        gender = bdata['gender']
        # print(f"Processing {filename}: Gender={gender}, FPS={fps}")
    except KeyError:
        print(f"Skipping {filename}: Missing metadata")
        return

    # 3. 降采样
    down_sample = int(fps / TARGET_FPS)
    if down_sample < 1: down_sample = 1
    
    # 提取 SMPL 参数
    # AMASS 数据格式通常为: poses (T, 156), trans (T, 3), betas (16)
    # 我们只取需要的帧
    poses = bdata['poses'][::down_sample]
    trans = bdata['trans'][::down_sample]
    num_frames = poses.shape[0]
    
    # 4. 准备 Tensor 输入
    bm = male_bm if gender == 'male' else female_bm
    
    body_parms = {
        'root_orient': torch.Tensor(poses[:, :3]).to(DEVICE),
        'pose_body': torch.Tensor(poses[:, 3:66]).to(DEVICE), # 只取身体部分
        'pose_hand': torch.Tensor(poses[:, 66:]).to(DEVICE),  # 手部参数
        'trans': torch.Tensor(trans).to(DEVICE),
        # betas 需要重复扩充到每一帧
        'betas': torch.Tensor(np.repeat(bdata['betas'][:10][np.newaxis], repeats=num_frames, axis=0)).to(DEVICE),
    }

    # 5. Forward Pass (SMPL Layer) -> 获得 3D 关节坐标
    with torch.no_grad():
        body = bm(**body_parms)
        # body.Jtr 包含了所有关节的位置 (T, 52, 3) 
        # HumanML3D 通常使用 SMPL 的前 22 个关节
        pose_seq_np = body.Jtr.detach().cpu().numpy()[:, :22, :]

    # 6. 坐标系转换 (根据原脚本逻辑)
    # 原脚本：np.dot(pose, trans_matrix)
    # trans_matrix = [[1, 0, 0], [0, 0, 1], [0, 1, 0]] ->  x=x, y=z, z=y (Y-up to Z-up swap? or vice versa)
    # 通常 AMASS 是 Z-up, HumanML3D 需要 Y-up
    # 原矩阵效果：New X = Old X, New Y = Old Z, New Z = Old Y
    # 让我们明确一下：通常图形学中 Y 是朝上的。
    trans_matrix = np.array([[1.0, 0.0, 0.0],
                             [0.0, 0.0, 1.0],
                             [0.0, 1.0, 0.0]])
    pose_seq_np_n = np.dot(pose_seq_np, trans_matrix)
    
    # 7. 减去中心位移 (可选，HumanML3D 后续处理会做，但这里可以先不做)
    # pose_seq_np_n -= pose_seq_np_n[0:1, 0:1, :] * np.array([1, 0, 1]) # Remove initial position on floor?
    
    # 8. 保存数据
    save_path = os.path.join(output_dir, filename + '.npy')
    np.save(save_path, pose_seq_np_n)
    
    # 9. 额外的详细信息打印和可视化 (针对第一个处理的文件)
    return pose_seq_np_n

def main():
    # 模拟命令行参数，你可以直接改这里
    input_folder = "./amass_sample/ACCAD" # 你的解压后的 AMASS 子数据集文件夹
    output_folder = "./output_joints"
    
    os.makedirs(output_folder, exist_ok=True)
    
    # 加载模型
    male_bm, female_bm = load_body_models()
    
    # 查找文件
    files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.npz')]
    print(f"Found {len(files)} .npz files in {input_folder}")
    
    if len(files) == 0:
        print("No files found. Please check input path.")
        return

    # 批处理
    first_processed_data = None
    first_filename = ""
    
    for f in tqdm(files):
        data = process_single_file(f, output_folder, male_bm, female_bm)
        if first_processed_data is None and data is not None:
            first_processed_data = data
            first_filename = os.path.basename(f)

    # ---------------- 可视化与检查 ----------------
    if first_processed_data is not None:
        print("\n" + "="*30)
        print(f"Details for {first_filename}")
        print("="*30)
        print(f"Shape: {first_processed_data.shape} -> (Frames, Joints, XYZ)")
        print(f"Frame count: {first_processed_data.shape[0]}")
        print(f"Joint count: {first_processed_data.shape[1]} (Should be 22)")
        
        # 打印某一帧的数值示例
        print("\nData Sample (Frame 0, First 5 joints):")
        print(first_processed_data[0, :5, :])
        
        # 保存可视化图片
        vis_save_path = os.path.join(output_folder, f"vis_{first_filename}.png")
        visualize_pose(first_processed_data, vis_save_path, frame_idx=0)
        
        print("\nDone! Check the output folder for .npy files and the visualization png.")

if __name__ == "__main__":
    main()
```

### 如何使用这个脚本？

1.  **下载 Body Models**: 确保你有 `./body_models/smplh/male/model.npz` 和 female 的对应文件。
2.  **准备数据**: 解压你的 AMASS zip 文件（例如 `ACCAD.tar.bz2` 解压到 `./amass_sample/ACCAD`）。
3.  **修改路径**: 修改脚本中 `input_folder` 指向你的数据文件夹。
4.  **运行**: `python convert_amass_single.py`。

### 总结

*   你提供的原始 notebook 仅仅完成了 **从 SMPL 参数还原成 3D 关节坐标 (N, 22, 3)** 的过程。
*   你心中疑惑的 **263 维** 是在这个步骤产生的 `.npy` 文件基础上，进一步计算速度、旋转、接触点等物理特征得到的。
*   上面的脚本帮你完成了第一步，并增加了数据检查和可视化，这对于理解数据流非常有帮助。转换完得到 `.npy` 后，如果想得到 263 维特征，通常需要运行 HumanML3D 仓库中的 `dataset/motion_process.py`。

在 AMASS 和 SMPL 协议中，`trans`（Translation）通常是**全局的世界空间坐标（World Space/Global Space）**。

为了让你彻底理解这个结构，我们可以从以下三个维度来看：

### 1. `trans` 的物理本质`trans` 表示的是 SMPL 模型**根关节点（Root Joint，通常位于盆骨位置）在全局坐标系中的绝对位置**。

* 如果 `trans = [0, 0, 0]`，意味着人体的中心点位于世界的原点。
* 如果人在跑步，`trans` 会随着帧数的变化而在空间中连续位移。

### 2. 坐标系的层级关系在 3D 角色动画中，坐标系是逐层嵌套的，你可以这样理解：

* **世界坐标系 (World Space)**：整个 3D 场景的固定坐标系。
* **根坐标系 (Root/Global Frame)**：由 `trans`（位置）和 `root_orient`（旋转）定义。它决定了角色“在哪”以及“面朝哪”。
* **局部坐标系 (Local Space/Joint Space)**：这是 `pose_body`（身体姿态）所在的坐标系。每一个骨骼关节（如膝盖、手肘）的旋转都是**相对于其父关节**的。

### 3. `trans` 与 `root_orient` 的组合效应当你把这些参数输入 SMPL 模型时，计算过程大致如下：

1. **姿态变换**：根据 `pose_body` 让静态的身体模型摆出动作（此时角色还在原点）。
2. **根部旋转**：应用 `root_orient`，让摆好动作的角色旋转到正确的朝向（此时角色依然在原点附近旋转）。
3. **全局平移**：最后应用 `trans`，将整个角色“整体搬运”到世界空间的某个 (x, y, z) 位置。

### 🚨 深度学习中的特例（重要提示）虽然 AMASS 原始数据里的 `trans` 是全局的，但在**训练动作生成模型（如你的 MCM-LDM）**时，研究者通常会做“归一化”处理：

* **重心归一化**：将每一段动作的第一帧 `trans` 减掉，让动作从原点开始。
* **局部化转换**：有些算法会将全局 `trans` 转换为**速度（Velocity）**。比如：`trans_velocity = trans[t] - trans[t-1]`。这样做是因为模型更容易学习“这一帧往前走了多远（局部感官）”，而不是学习“我现在在世界的坐标 (100, 200) 处”。

**总结：**
在你的 `body_parms` 字典里，`trans` 指的是**世界空间坐标**。但如果你发现模型生成的动作总是原地踏步，那可能是因为你的预处理脚本将 `trans` 减去了初始值，或者只保留了相对位移。

你需要我写一段代码，帮你可视化 `trans` 的轨迹，看看它是绕原点转圈还是在世界坐标系里乱跳吗？

你的理解非常直观，**`root_orient` 确实是身体整体的旋转角度**，它决定了人是面朝北还是面朝南，是站着还是躺着。

但是，在 AMASS 数据集和 SMPL 模型中，它的数学表示通常**不是欧拉角**，而是 **轴角（Axis-Angle）**。

---

### 1. 为什么不是欧拉角？虽然欧拉角（如：俯仰 Pitch、偏航 Yaw、翻滚 Roll）符合人类直觉，但在计算机图形学和深度学习中，它有两个致命缺点：

1. **万向节死锁 (Gimbal Lock)**：当某个轴旋转 90° 时，会丢失一个自由度。
2. **不唯一性**：同一个旋转可以用多组不同的欧拉角表示，这会让神经网络（如你的 DiT 模型）感到困惑，难以收敛。

### 2. `root_orient` 的真实面目：轴角 (Axis-Angle)在 `poses` 数组的前 3 维中，这三个值 (r_x, r_y, r_z) 构成了一个**向量**：

* **向量的方向**：代表旋转所绕的**旋转轴**。
* **向量的模长（长度）**：代表旋转的**角度大小**（弧度制）。

例如，如果 `root_orient` 是 `[0, 1.57, 0]`，意味着人体绕着 **Y 轴**（垂直轴）旋转了大约 **1.57 弧度**（即 90°）。

### 3. 在代码中如何转换？SMPL 模型的层（Layer）通常会自动处理这个轴角。如果你需要将其转换为其他格式（例如旋转矩阵或四元数），通常会用到 **罗德里格斯公式 (Rodrigues' rotation formula)**。

在 PyTorch 环境下，你经常会看到这样的转换逻辑：

```python
from pytorch3d.transforms import axis_angle_to_matrix, matrix_to_quaternion

# 将 root_orient (B, 3) 转换为旋转矩阵 (B, 3, 3)
root_matrix = axis_angle_to_matrix(body_parms['root_orient'])

```

### 总结| 维度 | 常见名称 | 数学格式 | 在 SMPL 中的作用 |
| --- | --- | --- | --- |
| **`trans`** | 位移 | 3D 向量 (x, y, z) | 决定盆骨在世界地图上的坐标。 |
| **`root_orient`** | 根旋转 | **3D 轴角** (\omega_x, \omega_y, \omega_z) | 决定身体整体的姿态朝向（相对于世界坐标系）。 |


## step 2：motion_representation ： https://github.com/EricGuo5513/HumanML3D/blob/main/motion_representation.ipynb
注意正确的代码在这里：/root/autodl-tmp/MyRepository/MCM-LDM/step2_full.py

太棒了！进入第二部分，这确实是 HumanML3D 最核心、也是最“数学”的部分。

这部分的目标是：**从原始的 3D 坐标 (Raw Joints) $\rightarrow$ 神经网络能“吃”的 263 维特征向量 (Motion Features)**。

为了完成这个任务，我们需要依赖 HumanML3D 官方仓库中的一些工具函数（特别是 IK 反向运动学求解器）。**请确保你的工作目录下有 `common` 文件夹和 `paramUtil.py` 文件**（这通常在 HumanML3D 仓库的根目录能找到）。

我们将流程拆解为三个独立的 Step，每个 Step 我都为你写好了函数和对应的可视化检查。

---

### Step 1: 骨架标准化 (Uniform Skeleton)

**为什么要做这一步？**
原始数据（AMASS）里，每个人都不一样高，腿长也不一样。如果直接训练，神经网络会混淆：它不知道这个动作是因为“人腿长”才迈得远，还是因为“动作幅度大”。
**解决方案**：把所有人的骨架强制“拉伸/缩放”成一个**标准骨架**。

**怎么做？**
1.  计算原始骨架的腿长比例。
2.  使用 **反向运动学 (Inverse Kinematics, IK)** 计算出关节的旋转角度。
3.  把这些旋转角度应用到一个**标准长度的骨架**上（Forward Kinematics, FK）。

#### 代码实现 (Step 1)

你需要创建一个 `step1_standardize.py`（或者直接在你的主脚本里定义）：

```python
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 必须依赖 HumanML3D 的工具库
try:
    from common.skeleton import Skeleton
    from paramUtil import t2m_raw_offsets, t2m_kinematic_chain
except ImportError:
    print("Error: 请确保 'common' 文件夹和 'paramUtil.py' 在当前目录下！")
    exit()

def step1_uniform_skeleton(raw_positions):
    """
    Step 1: 将不同体型的骨架统一为标准骨架
    Input: raw_positions (T, 22, 3)
    Output: new_joints (T, 22, 3)
    """
    # 1. 准备标准骨架的 Offset (骨长)
    # t2m_raw_offsets 是 HumanML3D 定义的标准人数据
    n_raw_offsets = torch.from_numpy(t2m_raw_offsets)
    kinematic_chain = t2m_kinematic_chain
    
    # 2. 初始化骨架工具
    # 这里的 device 用 cpu 即可，IK 解算不需要太强的 GPU
    src_skel = Skeleton(n_raw_offsets, kinematic_chain, 'cpu')
    
    # 3. 计算缩放比例 (基于腿长)
    # 原始骨架的第一帧
    positions = torch.from_numpy(raw_positions)
    src_offset = src_skel.get_offsets_joints(positions[0])
    
    # 定义哪几个点是腿 (Left: 1-4-7-10, Right: 2-5-8-11)
    # 这里的索引对应 HumanML3D 的定义
    l_idx1, l_idx2 = 5, 8 # Right leg parts (Lower leg, Foot)
    
    # 计算原始腿长
    src_leg_len = torch.abs(src_offset[l_idx1]).max() + torch.abs(src_offset[l_idx2]).max()
    # 计算目标标准腿长
    tgt_offset = n_raw_offsets
    tgt_leg_len = torch.abs(tgt_offset[l_idx1]).max() + torch.abs(tgt_offset[l_idx2]).max()
    
    scale_rt = tgt_leg_len / src_leg_len
    # print(f"Scaling Ratio: {scale_rt.item():.4f}")
    
    # 4. 调整根节点位置
    src_root_pos = positions[:, 0]
    tgt_root_pos = src_root_pos * scale_rt
    
    # 5. IK + FK (核心步骤)
    # 先用 IK 算出每一帧的关节旋转 (Quaternions)
    # face_joint_indx: [R_Hip, L_Hip, R_Shoulder, L_Shoulder]
    face_joint_indx = [2, 1, 17, 16] 
    quat_params = src_skel.inverse_kinematics_np(raw_positions, face_joint_indx)
    
    # 再用这些旋转应用到标准骨架上 (FK)
    src_skel.set_offset(n_raw_offsets)
    new_joints = src_skel.forward_kinematics_np(quat_params, tgt_root_pos.numpy())
    
    return new_joints

def visualize_step1_comparison(raw, uniform, frame_idx=0):
    """可视化对比：原始骨架(红) vs 标准骨架(蓝)"""
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    p1 = raw[frame_idx]
    p2 = uniform[frame_idx]
    
    # 简单的连线 (只画躯干和腿示意)
    chain = [[0,1],[1,4],[4,7],[7,10], [0,2],[2,5],[5,8],[8,11], [0,3],[3,6],[6,9]]
    
    for link in chain:
        ax.plot([p1[link[0],0], p1[link[1],0]], [p1[link[0],2], p1[link[1],2]], [p1[link[0],1], p1[link[1],1]], c='r', alpha=0.5)
        ax.plot([p2[link[0],0], p2[link[1],0]], [p2[link[0],2], p2[link[1],2]], [p2[link[0],1], p2[link[1],1]], c='b')
        
    ax.set_title("Red: Raw (Original Scale) | Blue: Uniform (Standard Scale)")
    ax.set_xlabel('X'); ax.set_ylabel('Z (Depth)'); ax.set_zlabel('Y (Height)')
    # 强制比例一致以便观察大小区别
    ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(0,2)
    plt.savefig("step1_vis.png")
    plt.close()
    print("Step 1 visualization saved to step1_vis.png")

```

---

### Step 2: 规范化与对齐 (Canonicalization)

**为什么要做这一步？**
我们希望模型学习的是“动作本身”，而不是“人在哪里”。
*   如果一个人在房间角落做广播体操，和在房间中心做，动作特征应该是完全一样的。
*   如果一个人面朝南做，和面朝北做，动作也应该是一样的。

**怎么做？**
1.  **Put on Floor**: 把动作的最低点（脚）放在地面上 (Y=0)。
2.  **Face Z+**: 旋转整个序列，让第一帧的人**面朝 Z 轴正方向**。
3.  **Root at Origin**: 把第一帧的根节点挪到 $(0,0,0)$。

#### 代码实现 (Step 2)

```python
from common.quaternion import qbetween_np, qrot_np

def step2_canonicalize(positions):
    """
    Step 2: 对齐与归一化
    Input: positions (T, 22, 3) -> 已经是标准骨架
    Output: aligned_positions (T, 22, 3)
    """
    positions = positions.copy()
    
    # 1. 放在地板上 (Put on Floor)
    # 找到所有帧、所有关节中最低的 Y 值
    floor_height = positions.min(axis=0).min(axis=0)[1]
    positions[:, :, 1] -= floor_height
    
    # 2. 根节点归零 (XZ at origin)
    # 此时只把第一帧的位移减掉，后续的位移是相对于第一帧的
    root_pos_init = positions[0]
    root_pose_init_xz = root_pos_init[0] * np.array([1, 0, 1]) # 只取 X, Z
    positions = positions - root_pose_init_xz
    
    # 3. 旋转到面朝 Z+ (Face Z+)
    # 定义关键点索引
    r_hip, l_hip = 2, 1
    sdr_r, sdr_l = 17, 16 # Shoulders
    
    # 计算第一帧的身体朝向向量
    # 向量1: 左右胯连线 (Right - Left)
    across1 = root_pos_init[r_hip] - root_pos_init[l_hip]
    # 向量2: 左右肩连线
    across2 = root_pos_init[sdr_r] - root_pos_init[sdr_l]
    # 综合两者的横向向量
    across = across1 + across2
    across = across / np.sqrt((across ** 2).sum(axis=-1))[..., np.newaxis]
    
    # 前向向量 = Y轴 (0,1,0) 叉乘 横向向量
    # 这里的 Y轴是 (0,1,0) 没问题，因为 HumanML3D 是 Y-up
    forward_init = np.cross(np.array([[0, 1, 0]]), across, axis=-1)
    forward_init = forward_init / np.sqrt((forward_init ** 2).sum(axis=-1))[..., np.newaxis]
    
    # 计算旋转四元数：把 forward_init 转到 [0, 0, 1] (Target Z+)
    target = np.array([[0, 0, 1]])
    root_quat_init = qbetween_np(forward_init, target)
    
    # 将这个旋转应用到所有帧、所有关节
    # root_quat_init 是 (1, 4), 需要广播到 (T, J, 4)
    root_quat_init = np.ones(positions.shape[:-1] + (4,)) * root_quat_init
    positions = qrot_np(root_quat_init, positions)
    
    return positions

def visualize_step2_alignment(aligned_data, frame_idx=0):
    """可视化检查：人应该站在原点，且面朝 Z 轴 (蓝色箭头方向)"""
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    p = aligned_data[frame_idx]
    
    # 画关节
    ax.scatter(p[:,0], p[:,2], p[:,1], s=20, c='k')
    
    # 画箭头指示
    # 红色=X, 绿色=Y(Up), 蓝色=Z(Forward)
    ax.quiver(0,0,0, 1,0,0, color='r', length=0.5, normalize=True)
    ax.quiver(0,0,0, 0,1,0, color='b', length=0.5, normalize=True) # Plot Y is Z
    ax.quiver(0,0,0, 0,0,1, color='g', length=0.5, normalize=True) # Plot Z is Y
    
    ax.set_title("Step 2: Should be at (0,0,0) facing Z (Blue Arrow)")
    ax.set_xlabel('X')
    ax.set_ylabel('Z (Depth)')
    ax.set_zlabel('Y (Height)')
    ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(0,2)
    
    plt.savefig("step2_vis.png")
    plt.close()
    print("Step 2 visualization saved to step2_vis.png")
```

---

### Step 3: 特征提取 (Feature Extraction)

**这是最复杂的一步，也就是 263 维的由来。**

此时我们有一个干净的、标准化的坐标数据 `(T, 22, 3)`。我们要把它转换成富含物理信息的特征 `(T, 263)`。


---

### 核心概念前置：三个坐标系的套娃

在讲具体的维度之前，必须先统一“参照系”。想象你身上绑了一个 GoPro 相机：

1.  **世界坐标系 (World)**: 绝对的东南西北。不管你怎么动，地面永远不动。
2.  **根节点坐标系 (Root-Centric)**: **这是最重要的！** 想象这个坐标系的原点永远粘在你的盆骨（Root）上，而且**永远面朝你身体的正前方**。
    *   如果你向前走，在这个坐标系里，你的相对位移是 0（因为坐标系跟着你走）。
    *   如果你伸出手，手的位置是相对于盆骨的。
3.  **父节点坐标系 (Parent-Local)**: 想象你的小臂是相对于大臂运动的。大臂不动，小臂弯曲，这就是父节点相对旋转。

**HumanML3D 的 263 维特征，绝大多数都是基于第 2 种（根节点坐标系）或第 3 种（父节点坐标系）的。** 这就是为了保证：不管你在哪里跳舞，动作的数据特征长得都一样。

---

### 263 维全景解构
公式：**4 (根) + 63 (位置) + 66 (速度) + 126 (旋转) + 4 (脚) = 263**

#### 第一部分：Root Data (根节点信息) —— [4 维]
这 4 个数字描述了“整个人”在地上是怎么移动的。

1.  **`r_velocity` (1维)**: **根节点角速度 (Angular Velocity)**
    *   **含义**：你转身转得有多快。
    *   **单位**：弧度/帧 (Radians per frame)。
    *   **轴向**：只记录绕 **Y轴 (垂直轴)** 的旋转。
    *   **直观理解**：如果你像陀螺一样原地旋转，这个值很大；如果你直着走，这个值是 0。

2.  **`l_velocity` (2维)**: **根节点线性速度 (Linear Velocity)**
    *   **含义**：你在地面上移动的速度。
    *   **坐标系**：**根节点坐标系** (Root-Centric)。
    *   **维度**：`[X轴速度, Z轴速度]`。
    *   **关键点**：这里的 X 和 Z **不是** 这里的东南西北，而是 **“相对于你当前面朝方向”** 的横向和纵向。
        *   `Velocity_Z`: 正数代表**向前**跑的速度，负数代表倒退。
        *   `Velocity_X`: 正数代表**向右**横移（侧步）的速度，负数向左。
    *   **回答你的疑问**：这里的速度是**线性速度 (米/帧)**，不是欧拉角。

3.  **`root_y` (1维)**: **根节点高度**
    *   **含义**：你的盆骨离地面的绝对高度。
    *   **作用**：区分是蹲着走、站着走还是跳在空中。

---

#### 第二部分：RIC Data (相对位置信息) —— [63 维]
RIC = Rotation Invariant Coordinates (旋转不变坐标)。

*   **含义**：**身体姿态**。也就是“不管你在哪，你的手在哪”。
*   **计算公式**：$P_{joint} - P_{root}$ (所有关节减去根节点位置)。
*   **坐标系**：**根节点坐标系**。
    *   这些坐标已经经过了旋转校正。不管你面朝南还是朝北，只要你摆出“大字形”，这些数值都是一样的。
*   **维度详解**：
    *   总共 22 个关节，减去根节点本身（因为在相对坐标系里根节点永远是 0,0,0，没必要存），剩下 21 个关节。
    *   每个关节有 (x, y, z) 3 个坐标。
    *   **21 × 3 = 63 维**。

---

#### 第三部分：Local Velocity (关节速度) —— [66 维]
这是你问得比较多的部分。

*   **含义**：每个关节运动的快慢和方向。
*   **是欧拉角吗？** **绝对不是！** 它是**线性速度** (Linear Velocity)，单位通常是 米/帧。即：上一帧到这一帧，这个关节移动了多少距离。
*   **相对于什么坐标系？** **根节点坐标系 (Root-Centric)**。
    *   这点非常重要！它排除了根节点的移动速度，只记录**肢体相对于躯干的挥动速度**。
    *   例子：如果你保持僵硬的姿势被传送带运走，你的 Root Velocity (第一部分) 很大，但这里的 Joint Velocity 全是 0。只有你挥手、踢腿时，这里才会有数值。
*   **维度详解**：
    *   包含根节点在内的所有 22 个关节。
    *   每个关节有 $(v_x, v_y, v_z)$ 3 个分量。
    *   **22 × 3 = 66 维**。

---

#### 第四部分：Rot Data (关节旋转 - 6D表示) —— [126 维]
这是最抽象的部分。

*   **含义**：每个关节是怎么**转**的（弯曲程度）。
*   **相对于谁？** **相对于父关节 (Parent-Joint)**。
    *   例如：左小臂的旋转是相对于左大臂的。如果大臂动、小臂锁死不动，那么小臂的相对旋转量就是 0（或者单位矩阵）。
*   **什么是 6D 连续表示 (6D Continuous Representation)？**
    *   **背景**：
        *   **欧拉角 (3数)**：有“万向节死锁”问题，且 0度和360度是同一个姿态但数值差巨大，神经网络很难学（不连续）。
        *   **四元数 (4数)**：解决了死锁，但有“双倍覆盖”问题（$q$ 和 $-q$ 代表同一个旋转），神经网络也很难学。
        *   **旋转矩阵 (9数)**：$3\times3$ 的矩阵，虽然精确但参数太多，且很难保证矩阵的正交性。
    *   **6D 表示法**：这是 CVPR 2019 的一篇论文提出的神技。它取 $3\times3$ 旋转矩阵的**前两列**（Column 1 和 Column 2）。
        *   矩阵 $R = [C_1, C_2, C_3]$。我们只存 $[C_1, C_2]$，也就是 6 个数。
        *   **为什么能还原？** 因为旋转矩阵是正交矩阵，列向量互相垂直且模为1。只要知道前两个向量，第三个向量可以通过**叉乘** ($C_3 = C_1 \times C_2$) 算出来！
    *   **优点**：这 6 个数在实数空间里是完全连续的，神经网络随便预测 6 个数，都能通过 Gram-Schmidt 正交化变成一个合法的旋转矩阵。这是目前动作生成的标准配置。
*   **维度详解**：
    *   21 个关节（通常根节点的旋转被单独处理了，或者包含在 Root Data 里，这里指身体关节）。
    *   每个关节 6 个参数。
    *   **21 × 6 = 126 维**。

---

#### 第五部分：Foot Contact (脚部接触) —— [4 维]
*   **含义**：脚有没有踩实地面。
*   **为什么需要？** 动作生成最怕“滑步” (Floating/Sliding)。如果模型不知道脚该不该动，它就会生成那种脚像在溜冰一样的动作。
*   **维度**：
    1.  左脚跟 (Left Heel) 是否着地 (0或1)
    2.  左脚尖 (Left Toe) 是否着地
    3.  右脚跟 (Right Heel) 是否着地
    4.  右脚尖 (Right Toe) 是否着地
*   通常这是一个概率值（Sigmoid输出），训练时用 0/1 监督。

---

### 总结图谱

为了让你彻底明白，想象一个“挥手”的动作：

1.  **Root Data (4)**: 你站着没走，所以线速度为0；没转身，角速度为0；高度恒定。
2.  **RIC Data (63)**: 你的手在这一帧相对于你盆骨的坐标位置（比如在右上方）。
3.  **Local Velocity (66)**: 你的手正在向右上方快速移动，速度向量比如是 $(0.5, 0.5, 0)$。
4.  **Rot Data (126)**: 你的肩膀相对于躯干旋转了多少度，肘关节相对于大臂旋转了多少度（用6个数字描述这个角度）。
5.  **Foot Contact (4)**: 你的两只脚都稳稳踩在地上，全是 1。

这就是 263 维特征向量里的秘密！它把一个动作从“绝对空间”剥离出来，变成了“纯粹的身体运动规律”。

#### 代码实现 (Step 3)

```python
from common.skeleton import Skeleton
from common.quaternion import *
from paramUtil import *

def step3_extract_features(positions):
    """
    Step 3: 计算 263 维特征
    Input: positions (T, 22, 3) -> 已经是 Step 2 处理过的
    Output: feature_vector (T, 263)
    """
    # ---------------- A. 脚部接触 (Foot Contacts) ----------------
    # 阈值
    fid_r, fid_l = [8, 11], [7, 10] # 右脚(跟/尖), 左脚(跟/尖)
    velfactor = np.array([0.002, 0.002]) # 速度阈值
    
    # 计算两帧之间的速度平方
    # Left Feet
    feet_l_vel = (positions[1:, fid_l] - positions[:-1, fid_l]) ** 2
    feet_l_vel = feet_l_vel.sum(axis=-1) # (T-1, 2)
    feet_l = (feet_l_vel < velfactor).astype(np.float32)
    
    # Right Feet
    feet_r_vel = (positions[1:, fid_r] - positions[:-1, fid_r]) ** 2
    feet_r_vel = feet_r_vel.sum(axis=-1)
    feet_r = (feet_r_vel < velfactor).astype(np.float32)
    
    # ---------------- B. 旋转与根节点信息 (IK Again) ----------------
    n_raw_offsets = torch.from_numpy(t2m_raw_offsets)  # 这是一个简化的模型，其实没有T Pose，这个t2m_raw_offsets就是比如子joint默认在父joint的哪个方向而已
    kinematic_chain = t2m_kinematic_chain
    skel = Skeleton(n_raw_offsets, kinematic_chain, "cpu")
    
    # IK 解算四元数 (T, 22, 4)
    face_joint_indx = [2, 1, 17, 16]
    quat_params = skel.inverse_kinematics_np(positions, face_joint_indx, smooth_forward=True)
    
    # 提取根节点旋转 (Root Rotation)
    # 这里的 r_rot 是面向 Z+ 的校准旋转
    r_rot = quat_params[:, 0].copy() 
    
    # 1. Root Linear Velocity (地面线速度)
    # 计算每帧之间的位移
    velocity = (positions[1:, 0] - positions[:-1, 0]).copy()
    # 关键：要把这个速度旋转回“当前朝向”的局部坐标系
    # 这样速度就变成了 "向前走的速度" 和 "向侧面走的速度"，而不是 "向北走"
    velocity = qrot_np(r_rot[1:], velocity)
    l_velocity = velocity[:, [0, 2]] # 只取 X, Z (2维)
    
    # 2. Root Angular Velocity (角速度)
    # 计算两帧之间转了多少度
    r_velocity = qmul_np(r_rot[1:], qinv_np(r_rot[:-1]))
    r_velocity = np.arcsin(r_velocity[:, 2:3]) # 提取 Y 轴分量 (1维)
    
    # 3. Root Height
    root_y = positions[:, 0, 1:2] # (T, 1)
    
    # ---------------- C. 关节旋转 (Rot Data) ----------------
    # 将四元数转换为 6D 表示 (更适合神经网络)
    # 并且去掉根节点，只保留 21 个关节
    # quaternion_to_cont6d_np 是 HumanML3D 的工具函数
    cont_6d_params = quaternion_to_cont6d_np(quat_params)
    rot_data = cont_6d_params[:, 1:].reshape(len(cont_6d_params), -1) # (T, 21*6)
    
    # ---------------- D. 局部关节位置 (RIC Data) ----------------
    # 所有关节减去根节点位置
    positions_local = positions.copy()
    positions_local[..., 0] -= positions[:, 0:1, 0]
    positions_local[..., 2] -= positions[:, 0:1, 2]
    
    # 同样要旋转回当前的局部朝向
    # np.repeat 扩充维度以便计算
    positions_local = qrot_np(np.repeat(r_rot[:, None], positions.shape[1], axis=1), positions_local)
    
    # 去掉根节点 (因为全是0了)
    ric_data = positions_local[:, 1:].reshape(len(positions), -1) # (T, 21*3)
    
    # ---------------- E. 关节速度 (Local Velocity) ----------------
    # 全局速度
    global_vel = positions[1:] - positions[:-1]
    # 转为局部速度
    local_vel = qrot_np(np.repeat(r_rot[:-1, None], positions.shape[1], axis=1), global_vel)
    local_vel = local_vel.reshape(len(local_vel), -1) # (T-1, 22*3)
    
    # ---------------- 拼接 (Concatenation) ----------------
    # 注意长度对齐：速度相关特征会导致少一帧 (T-1)
    # 所以我们取 data[:-1]
    
    data = np.concatenate([
        r_velocity,       # (T-1, 1)
        l_velocity,       # (T-1, 2)
        root_y[:-1],      # (T-1, 1)
        ric_data[:-1],    # (T-1, 63)
        rot_data[:-1],    # (T-1, 126)
        local_vel,        # (T-1, 66)
        feet_l,           # (T-1, 2)
        feet_r            # (T-1, 2)
    ], axis=-1)
    
    return data # (T-1, 263)

def verify_feature_extraction(final_data, original_step2_data):
    """
    验证：通过 263 维特征尝试还原动作，看是否和 Step 2 的数据一致。
    如果一致，说明特征提取是对的。
    """
    # 这里涉及反向恢复函数 recover_from_ric (ipynb里有提供)
    # 为了简化，我们只检查维度
    print("-" * 30)
    print("Verification:")
    print(f"Original Frames: {original_step2_data.shape[0]}")
    print(f"Feature Vector:  {final_data.shape}")
    
    expected_dim = 1 + 2 + 1 + 63 + 126 + 66 + 4
    print(f"Calculated Dim: {expected_dim}")
    
    if final_data.shape[1] == 263:
        print("SUCCESS: Dimension is 263!")
    else:
        print(f"ERROR: Dimension mismatch! Got {final_data.shape[1]}")
```

### 总结与运行方式

你可以创建一个主文件 `pipeline.py`，把上面的代码粘进去，然后：

```python
if __name__ == "__main__":
    # 1. 加载我们在第一部分生成的 .npy (22, 3)
    raw_data = np.load("./output_joints/test_motion.npy")
    print(f"Loaded Raw Data: {raw_data.shape}")
    
    # Step 1
    uniform_data = step1_uniform_skeleton(raw_data)
    visualize_step1_comparison(raw_data, uniform_data)
    print("Step 1 Done: Skeleton Standardized.")
    
    # Step 2
    aligned_data = step2_canonicalize(uniform_data)
    visualize_step2_alignment(aligned_data)
    print("Step 2 Done: Motion Canonicalized.")
    
    # Step 3
    feature_vec = step3_extract_features(aligned_data)
    verify_feature_extraction(feature_vec, aligned_data)
    
    # 保存结果
    np.save("final_features_263.npy", feature_vec)
    print("All Done! Saved 'final_features_263.npy'")
```

### 关键点总结
1.  **Step 1** 解决了“腿长不同”的问题。
2.  **Step 2** 解决了“初始位置不同”的问题。
3.  **Step 3** 将绝对坐标变成了相对特征（速度、旋转），这让神经网络更容易学习动作的本质规律（比如“走路”就是双腿交替摆动，跟你在哪里走没关系）。

快去试试吧！如果你的环境里有 `common` 文件夹，这套代码应该能直接跑通。

## 关于归一化的逻辑以及如何可逆复原？
你观察得非常仔细！这说明你已经深入到了代码逻辑的细节中。这两个问题都直指 HumanML3D 处理流程的核心：**如何确保数据在“变换”和“特征化”之后依然保持物理真实性。**

我们来逐一拆解：

---

### 【1】关于“标准骨架”的加载逻辑

你的直觉是对的。在之前的简略描述中，我可能没把“标准骨架到底在哪”说透。

在 HumanML3D 的代码中，标准骨架并不是加载一个 `.npz` 或 `.obj` 模型文件，而是通过 **`paramUtil.py`** 中的两个硬编码常量定义的：

1.  **`t2m_raw_offsets`**: 这是一个形状为 `(22, 3)` 的数组。它定义了 22 个关节在**标准姿态（T-Pose）**下，每个关节相对于其父节点的偏移向量（也就是骨头长度和方向）。
2.  **`t2m_kinematic_chain`**: 定义了这 22 个关节是怎么连的（谁是谁的爸爸）。

**加载与应用逻辑如下：**
*   当你初始化 `Skeleton(n_raw_offsets, ...)` 时，你就创建了一个“标准体型”的骨架对象。
*   在 `uniform_skeleton` 函数里，代码先计算原始动作的腿长，算出缩放比例。
*   **关键点**：`quat_params = src_skel.inverse_kinematics_np(...)` 这一步得到的四元数，代表了**关节的旋转角度**。旋转角度是与体型无关的（比如“胳膊弯曲 90 度”这个信息在小孩和巨人身上是一样的）。
*   最后，`src_skel.set_offset(target_offset)` 将骨架的骨长强行设为 `t2m_raw_offsets`（标准值），再把刚才算出的旋转角度套上去。这样，动作没变，但人变成了“标准身材”。

---

### 【2】Step 3 核心：使用 `recover_from_ric` 进行闭环验证

仅仅维度对上（263维）确实不能说明逻辑正确。在 HumanML3D 论文中，作者强调了这套表示法是 **“可逆的”**。

`recover_from_ric` 的作用就是：从那 263 个看起来乱七八糟的数字中，把 **(N, 22, 3) 的 3D 坐标还原回来**。如果还原后的坐标和 Step 2 的坐标重合，就证明特征提取逻辑 100% 正确。

下面是为您重写的 **Step 3 验证逻辑**，包含完整的还原代码和误差计算：

#### 代码实现：`feature_verification.py`

```python
import numpy as np
import torch
from common.quaternion import qrot, qinv

def recover_from_ric(data, joints_num):
    """
    从 263 维特征中还原 3D 关节坐标 (RIC 路径)
    data: (Frames, 263)
    joints_num: 22
    """
    # 转换成 Tensor 方便计算
    if isinstance(data, np.ndarray):
        data = torch.from_numpy(data)
    
    # 1. 拆解特征向量
    # 0:1 旋转角速度, 1:3 平面线速度, 3:4 根节点高度
    r_vel = data[..., 0]      
    l_vel = data[..., 1:3]    
    root_y = data[..., 3]     
    # 4:4+63 局部位置数据 (RIC)
    ric_data = data[..., 4: (joints_num - 1) * 3 + 4]
    ric_data = ric_data.view(ric_data.shape[:-1] + (-1, 3)) # (T, 21, 3)

    # 2. 还原根节点朝向 (旋转积分)
    # 我们需要通过角速度累加，算出每一帧相对于第一帧转了多少度
    r_rot_ang = torch.zeros_like(r_vel)
    r_rot_ang[..., 1:] = r_vel[..., :-1]
    r_rot_ang = torch.cumsum(r_rot_ang, dim=-1) # 角度累加

    # 构造绕 Y 轴旋转的四元数
    r_rot_quat = torch.zeros(data.shape[:-1] + (4,))
    r_rot_quat[..., 0] = torch.cos(r_rot_ang * 0.5) # 四元数公式
    r_rot_quat[..., 2] = torch.sin(r_rot_ang * 0.5)

    # 3. 还原根节点位移 (位置积分)
    r_pos = torch.zeros(data.shape[:-1] + (3,))
    r_pos[..., 1:, [0, 2]] = l_vel[..., :-1] # 填充线速度
    # 关键：线速度是在局部坐标系的，还原到全局需要旋转回来
    # qrot(qinv(q), v) 是把局部速度转回世界坐标
    r_pos = qrot(qinv(r_rot_quat), r_pos)
    r_pos = torch.cumsum(r_pos, dim=-2) # 位移累加
    r_pos[..., 1] = root_y # 填充高度

    # 4. 还原局部关节到世界坐标
    # ric_data 是相对于根节点的，且在根节点局部坐标系下
    # a. 旋转回来
    # r_rot_quat 需要扩充维度匹配 joints (21个)
    batch_quat = r_rot_quat[..., None, :].expand(ric_data.shape[:-1] + (4,))
    positions = qrot(qinv(batch_quat), ric_data)
    
    # b. 加上根节点的位移
    positions[..., 0] += r_pos[..., 0:1]
    positions[..., 2] += r_pos[..., 2:3]
    # Y 轴直接加 (高度已经是相对地面的了，ric_data 的 Y 也是相对 root 的)
    positions[..., 1] += r_pos[..., 1:2]

    # 5. 合并根节点和肢体节点
    # 把根节点插回 index 0 的位置
    full_positions = torch.cat([r_pos.unsqueeze(-2), positions], dim=-2)

    return full_positions

def do_step3_and_verify(aligned_data):
    """
    完整的 Step 3 闭环测试
    aligned_data: (T, 22, 3) -> Step 2 的输出
    """
    from pipeline_logic import step3_extract_features # 假设你的提取函数在这里
    
    # --- 1. 特征提取 ---
    feature_vec = step3_extract_features(aligned_data) # 输出 (T-1, 263)
    
    # --- 2. 逆向还原 ---
    recovered_joints = recover_from_ric(feature_vec, 22) # 输出 (T-1, 22, 3)
    
    # --- 3. 误差对比 ---
    # 因为特征提取少了一帧，我们对比前 T-1 帧
    ground_truth = aligned_data[:-1] 
    
    # 计算平均欧式距离 (Mean Euclidean Error)
    diff = np.linalg.norm(recovered_joints.numpy() - ground_truth, axis=-1)
    mean_error = diff.mean()
    
    print("\n" + "="*40)
    print("STEP 3 VERIFICATION RESULT")
    print("="*40)
    print(f"Feature Vector Shape:   {feature_vec.shape}")
    print(f"Recovered Joints Shape: {recovered_joints.shape}")
    print(f"Mean Recovery Error:    {mean_error:.8f} meters")
    
    if mean_error < 1e-5:
        print(">>> SUCCESS: Feature representation is lossless!")
    else:
        print(">>> WARNING: High recovery error. Check your extraction logic.")
    print("="*40)
    
    return feature_vec, recovered_joints
```

### 为什么这个验证是“断案”的关键？

这 263 维中有两套数据描述姿态：
1.  **RIC (63维)**: 它是直接的局部 3D 坐标。
2.  **ROT (126维)**: 它是关节的旋转角度。

**HumanML3D 为什么选 RIC 来还原？**
因为从旋转角（ROT）还原回坐标（FK）会产生**误差累积**。如果肩膀转错 1 度，手指就会偏出好几厘米。
而从 RIC 还原只需要：**“旋转整体 + 平移整体”**。
*   如果你的 `l_velocity` 提取错了，还原出来的人会“在原地踏步”或“乱飞”。
*   如果你的 `r_velocity` 提取错了，还原出来的人走出的路径（轨迹）会弯曲。
*   如果你的 `ric_data` 提取坐标轴搞反了，还原出来的人肢体就会像被打断了一样扭曲。

### 总结 263 维的本质：

当你看着还原出来的 3D 小人依然能完美地走、完美地跳，你就知道这 263 个数字已经成功地把：
*   **动作的节奏**（速度分量）
*   **身体的姿态**（局部坐标和旋转）
*   **脚下的物理约束**（接触点）
*   **体型的标准化**（骨长对齐）

全部压缩在了一起。这就是为什么它是 Text-to-Motion 任务中**最强大、最通用**的输入表示。

下一步，如果你想看还原后的视频，可以直接把 `recovered_joints` 丢进我们之前写的 `render_video_humanml3d_style` 函数里。如果视频看起来和原始的一模一样，逻辑就彻底通了！